<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'> The Finance Semantic Layer: Cubes, Prompts, Glossary and the Database Tool Registry on the Teradata MCP Server
      
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>


<p style="font-size:20px;font-family:Arial"><b>Introduction:</b></p>

<p style="font-size:16px;font-family:Arial">
This is the third demo in the governed finance agent series. The
<a href="https://nbviewer.org/urls/storage.googleapis.com/trial_teradata_documents/TeradataCloud/UseCases/Governed_Langchain_Teradata_Agent/POP_Langchain_Teradata_MCP_Agent.ipynb" target="_blank">Governed LangChain Teradata Agent</a>
demo constrained <i>what the agent is allowed to do</i> with profiles and pre-approved SQL tools. The
<b>Observable Governed Agent Hooks</b> demo added a server-side flight recorder to measure <i>what the agent
actually did</i>. This notebook addresses the remaining trust problem, and it sits upstream of both:
</p>

<p style="font-size:18px;font-family:Arial; font-style:italic; margin-left:20px;">
Every time an LLM writes SQL against your schema, it is guessing: at table structure, at join logic, at metric
definitions, at business terminology. Guesses vary between runs. A semantic layer removes the guessing.
</p>

<p style="font-size:16px;font-family:Arial">
Following the
<a href="https://github.com/Teradata/teradata-mcp-server/blob/main/docs/server_guide/CUSTOMIZING.md" target="_blank">Teradata MCP Server Customizing guide</a>,
we declaratively define a <b>domain-focused semantic layer</b> for the finance loyalty domain. Everything is YAML
placed in a config directory, no Python code and no server changes. The layer consists of:
</p>

<table style="font-size:15px;font-family:Arial; border-collapse:collapse; margin-left:20px;">
  <tr style="">
    <th style="padding:6px 14px; text-align:left;">Object</th>
    <th style="padding:6px 14px; text-align:left;">What it does</th>
    <th style="padding:6px 14px; text-align:left;">Trust contribution</th>
  </tr>
  <tr><td style="padding:6px 14px;"><b>Tools</b></td><td style="padding:6px 14px;">Parameterized, pre-approved SQL exposed as callable MCP tools</td><td style="padding:6px 14px;">The model fills parameters; it never writes SQL</td></tr>
  <tr style=""><td style="padding:6px 14px;"><b>Cubes</b></td><td style="padding:6px 14px;">Semantic model of metrics and dimensions; the server compiles SQL at runtime</td><td style="padding:6px 14px;">One governed definition of every metric; deterministic SQL generation; invalid requests fail before reaching the database</td></tr>
  <tr><td style="padding:6px 14px;"><b>Prompts</b></td><td style="padding:6px 14px;">Predefined prompts served by the MCP server itself</td><td style="padding:6px 14px;">Even the agent's instructions become governed, versioned server configuration</td></tr>
  <tr style=""><td style="padding:6px 14px;"><b>Glossary</b></td><td style="padding:6px 14px;">Domain terms, definitions and synonyms, auto-enriched with references from cubes and tools</td><td style="padding:6px 14px;">The model and the business speak the same vocabulary</td></tr>
  <tr><td style="padding:6px 14px;"><b>Profiles</b></td><td style="padding:6px 14px;">Named sets of tools, prompts and resources per user group or use case</td><td style="padding:6px 14px;">Domain-specific server instantiations from one configuration</td></tr>
  <tr style=""><td style="padding:6px 14px;"><b>Database Tool Registry</b></td><td style="padding:6px 14px;">Tools registered as database objects (views, macros, UDFs) via registry views</td><td style="padding:6px 14px;">Tool definitions live with the data, governed by database permissions and per-user filtering</td></tr>
</table>

<p style="font-size:16px;font-family:Arial; margin-top:15px;">
The flow of this notebook: define the semantic layer in YAML (Section 3), explore each object type through direct
MCP calls to see exactly what the server does (Section 4), hand the layer to a LangChain agent (Section 5), and
finally register a tool <b>in the database itself</b> through the Tool Registry with per-user filtering (Section 6).
</p>

<div class="alert alert-block alert-warning">
<p style="font-size:16px;font-family:Arial">
<b>How this composes with the previous demos:</b> the semantic layer defines the governed vocabulary and query
surface. Profiles restrict who gets which parts of it. Server hooks record how it is used. All three are
declarative server-side mechanisms; the agent code stays thin and none of the trust guarantees depend on the model
behaving well.
</p>
</div>

<p style="font-size:16px;font-family:Arial">
<b>References:</b><br>
&bull; <a href="https://github.com/Teradata/teradata-mcp-server/blob/main/docs/server_guide/CUSTOMIZING.md" target="_blank">Teradata MCP Server - Customizing guide (semantic layers)</a><br>
&bull; <a href="https://github.com/Teradata/teradata-mcp-server/blob/main/docs/developer_guide/REGISTRY_IMPLEMENTATION.md" target="_blank">Database Tool Registry - implementation guide</a><br>
&bull; <a href="https://github.com/Teradata/teradata-mcp-server/blob/main/examples/server-customisation/registry_setup.sql" target="_blank">Registry setup - example SQL</a><br>
&bull; <a href="https://github.com/Teradata/jupyter-demos/blob/main/TeradataCloud/UseCases/Governed_Langchain_Teradata_Agent/Langchain_Teradata_MCP_Agent.ipynb" target="_blank">Governed LangChain Teradata Agent (companion demo)</a>
</p>


<hr style='height:2px;border:none'>
<b style = 'font-size:20px;font-family:Arial'>1. Configure the environment</b>

In [ ]:
%%capture
!pip install git+https://github.com/Teradata/teradata-mcp-server.git

In [ ]:
%%capture
!pip install -U langchain-mcp-adapters langchain langchain-openai --quiet

<div class="alert alert-block alert-info">
<p style = 'font-size:16px;font-family:Arial'><b>Please</b><i> restart the kernel after executing the above cell to include/update these libraries into memory for this kernel. The simplest way to restart the Kernel is by typing zero zero: <b> 0 0</b></i> and then clicking <b>Restart</b>.</p>
</div>

In [ ]:
import asyncio, sys, os
import json
import time
from getpass import getpass
from teradataml import *
from teradataml import create_context, set_auth_token
from teradataml import execute_sql
from dotenv import load_dotenv, dotenv_values
import pandas as pd
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.messages import HumanMessage


<hr style="height:2px;border:none">

<p style="font-size:20px;font-family:Arial">
  <b>2. Connect to TeradataCloud</b>
</p>

<p style="font-size:16px;font-family:Arial">
As in the companion demos, we connect to TeradataCloud so that <code>teradataml</code> and the MCP server can
operate against the database.
</p>

<p style="font-size:16px;font-family:Arial">
<b>Required parameters:</b><br>
&bull; <b>host</b>: Your Teradata system address<br>
&bull; <b>username / password</b>: Database credentials<br>
&bull; <b>base_url</b>: UES API endpoint for your Teradata environment<br>
&bull; <b>pat_token</b>: Personal Access Token for API authentication<br>
&bull; <b>pem_file</b>: SSL certificate file for secure connections
</p>

<p style = 'font-size:18px;font-family:Arial;'><b>2.1 Load the Environment Variables and Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial;'>Load the environment variables from a .env file and use them to create a connection context to TeradataCloud.</p>

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=Finance_Semantic_Layer.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<p style = 'font-size:18px;font-family:Arial;'><b>2.2 Set your Authentication Token for this session.</b></p>

In [ ]:
# We've already loaded all the values into our environment variables and into a dictionary, env_vars.

if set_auth_token(base_url=env_vars.get("ues_uri"),
                  pat_token=env_vars.get("access_token"), 
                  pem_file=env_vars.get("pem_file"),
                  valid_from=int(time.time())
                 ):
    print("UES Authentication successful")
else:
    print("UES Authentication failed. Check credentials.")
    sys.exit(1)

<p style = 'font-size:18px;font-family:Arial;'><b>2.3 A quick look at the domain data</b></p>
<p style = 'font-size:16px;font-family:Arial;'>The whole semantic layer is built over a single table, <code>DEMO_Financial.Business</code>. No new data is needed: the point of a cube is that even one table supports many governed metric and dimension combinations.</p>

In [ ]:
businesses = DataFrame(in_schema("DEMO_Financial", "Business"))
businesses

<hr style="height:2px;border:none">
<p style="font-size:20px; font-family:Arial"><b>3. Define the Semantic Layer, Declaratively</b></p>

<p style="font-size:16px; font-family:Arial">
One YAML file, <code>finance_objects.yml</code>, defines the entire layer: two governed tools, one cube, one
server-side prompt, and a glossary. A second file, <code>profiles.yml</code>, groups them into profiles. Editing
these files and restarting the server is the whole change-management story: add a metric, fix a definition,
retire a tool, all without touching agent code or server code.
</p>

<p style="font-size:18px; font-family:Arial"><b>3.1 Create the configuration directory</b></p>

In [ ]:
BASE_DIR   = "/home/jovyan/JupyterLabRoot/TeradataCloud/UseCases/Finance_Semantic_Layer"
CONFIG_DIR = os.path.join(BASE_DIR, "mcp_config")
os.makedirs(CONFIG_DIR, exist_ok=True)
print("Config directory ready:", CONFIG_DIR)

<p style="font-size:18px; font-family:Arial"><b>3.2 The semantic layer definition: <code>finance_objects.yml</code></b></p>

<p style="font-size:16px; font-family:Arial">
The centerpiece is the <code>finance_business_portfolio</code> <b>cube</b>. Read it as a contract: these are the
only dimensions, the only metric definitions, and the only base row set the organization endorses for portfolio
analysis. Note four things in the definition:
</p>

<ul style="font-size:16px; font-family:Arial; margin-left:20px; line-height:1.8;">
  <li><b>Measures encode policy.</b> <code>base_eligible_count</code> is the loyalty policy's base eligibility clause (Active, Enterprise, 12+ months tenure, no risk flag) written once, by the data team, instead of re-derived by an LLM on every question.</li>
  <li><b>An optional join.</b> The <code>region_totals</code> join (a SQL subquery) is materialized by the compiler <i>only</i> when a request selects the <code>region_size</code> dimension. Requests that do not need it never pay for it.</li>
  <li><b>A nullable optional parameter.</b> <code>status_filter</code> narrows the base row set when provided; blank values and placeholders such as <code>NULL</code>, <code>None</code> or <code>N/A</code> are treated as SQL NULL, meaning no filter.</li>
  <li><b>Two filter phases.</b> Callers get <code>filter</code> (before aggregation, may reference any cube dimension) and <code>res_filter</code> (after aggregation, references computed measures). We demonstrate both in Section 4.</li>
</ul>

In [ ]:
%%writefile /home/jovyan/JupyterLabRoot/TeradataCloud/UseCases/Finance_Semantic_Layer/mcp_config/finance_objects.yml
# ---------------------------------------------------------------
# Governed tools: parameterized, pre-approved SQL
# ---------------------------------------------------------------
finance_check_business_loyalty_eligibility:
  type: tool
  description: Return business attributes needed to evaluate loyalty discount eligibility (status, type, region, risk, tenure) for one business.
  parameters:
    business_name:
      type: string
      description: Exact business_name as stored in DEMO_Financial.Business.
      required: true
  sql: |
    SELECT
      business_id,
      business_name,
      business_type,
      region,
      status,
      risk_flag,
      ((EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM start_date)) * 12
        + (EXTRACT(MONTH FROM CURRENT_DATE) - EXTRACT(MONTH FROM start_date))) AS tenure_months
    FROM "DEMO_Financial"."Business"
    WHERE business_name = :business_name;

finance_list_businesses:
  type: tool
  description: Fuzzy lookup of business names. Use only when an exact business_name lookup returned 0 rows, to recover the correct spelling before asking the user.
  parameters:
    name_fragment:
      type: string
      description: Partial business name to match, case-insensitive.
      required: true
  sql: |
    SELECT
      business_id,
      business_name,
      business_type,
      region,
      status
    FROM "DEMO_Financial"."Business"
    WHERE LOWER(business_name) LIKE LOWER('%' || :name_fragment || '%');

# ---------------------------------------------------------------
# Cube: the governed semantic model for portfolio analysis
# ---------------------------------------------------------------
finance_business_portfolio:
  type: cube
  description: Business portfolio metrics by region, business type, status and risk. The single governed source for portfolio counts, risk posture, tenure and base loyalty eligibility.
  sql: |
    SELECT
      business_id,
      business_name,
      business_type,
      region,
      status,
      risk_flag,
      ((EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM start_date)) * 12
        + (EXTRACT(MONTH FROM CURRENT_DATE) - EXTRACT(MONTH FROM start_date))) AS tenure_months
    FROM "DEMO_Financial"."Business"
    WHERE COALESCE(:status_filter, status) = status
  joins:
    - name: region_totals
      sql: (SELECT region AS r, COUNT(*) AS region_count FROM "DEMO_Financial"."Business" GROUP BY region)
      on: "finance_business_portfolio.region = region_totals.r"
      type: left
      optional: true
  dimensions:
    region:
      description: Business region, for example NA or EMEA.
      expression: finance_business_portfolio.region
    business_type:
      description: Business type, for example Enterprise, SMB or Gov.
      expression: finance_business_portfolio.business_type
    status:
      description: Account status, for example Active or Inactive.
      expression: finance_business_portfolio.status
    risk_flag:
      description: Risk marker, Y or N. Risk-flagged businesses are excluded from loyalty discounts.
      expression: finance_business_portfolio.risk_flag
    region_size:
      description: Total number of businesses in the same region, from the optional region_totals join.
      expression: region_totals.region_count
  measures:
    business_count:
      description: Number of businesses.
      expression: COUNT(*)
    risk_flagged_count:
      description: Number of businesses carrying the risk flag (excluded from loyalty discounts by policy).
      expression: SUM(CASE WHEN finance_business_portfolio.risk_flag = 'Y' THEN 1 ELSE 0 END)
    avg_tenure_months:
      description: Average tenure in months since contract start.
      expression: AVG(finance_business_portfolio.tenure_months)
    base_eligible_count:
      description: Businesses passing the base loyalty eligibility clause of policy v2 (Active, Enterprise, 12+ months tenure, no risk flag).
      expression: SUM(CASE WHEN finance_business_portfolio.status = 'Active' AND finance_business_portfolio.business_type = 'Enterprise' AND finance_business_portfolio.tenure_months >= 12 AND finance_business_portfolio.risk_flag = 'N' THEN 1 ELSE 0 END)
  parameters:
    status_filter:
      description: Optional exact status filter for the base row set, for example Active. Blank or NULL means all statuses.
      optional: true

# ---------------------------------------------------------------
# Prompt: served by the MCP server itself
# ---------------------------------------------------------------
finance_portfolio_analyst:
  type: prompt
  description: Governed analyst prompt for the finance semantic layer.
  prompt: "You are a finance portfolio analyst. Answer questions using ONLY the governed semantic layer tools. For aggregate questions (counts, averages, shares, comparisons) call the finance_business_portfolio cube tool with the published dimension names (region, business_type, status, risk_flag, region_size) and measure names (business_count, risk_flagged_count, avg_tenure_months, base_eligible_count). For questions about a single business call finance_check_business_loyalty_eligibility, and use finance_list_businesses only to recover a misspelled name. Never write raw SQL. State which measures and dimensions you used in every answer, using the exact published names."

# ---------------------------------------------------------------
# Glossary: shared vocabulary, auto-enriched from cubes and tools
# ---------------------------------------------------------------
glossary:
  type: glossary
  loyalty discount:
    definition: A 10% discount granted to businesses meeting the loyalty policy v2 base clause of Active status, Enterprise type, at least 12 months tenure, and no risk flag.
    synonyms:
      - loyalty rebate
      - retention discount
  risk flag:
    definition: A Y/N marker on a business. Risk-flagged businesses are excluded from loyalty discounts regardless of tenure or business type.
    synonyms:
      - risk indicator
      - risk marker
  tenure:
    definition: Months elapsed between a business contract start_date and today. Businesses with less than 12 months tenure are not eligible for loyalty discounts.
    synonyms:
      - contract age
      - customer age


<p style="font-size:18px; font-family:Arial"><b>3.3 Profiles: <code>profiles.yml</code></b></p>

<p style="font-size:16px; font-family:Arial">
Two profiles for two moments in this notebook. <code>finance_semantic</code> exposes the YAML semantic layer.
<code>finance_registry</code> adds the <code>registry</code> key for Section 6: it names the database schema
containing the registry views (here, your own database), and registry tools are loaded from there on the first
database connection. Profiles use regular expression patterns, and top-level keys in this file completely override
any packaged profile of the same name. We write this file with Python rather than <code>%%writefile</code> because
the registry schema is your username.
</p>

In [ ]:
username = env_vars.get("username")

profiles_yaml = f"""finance_semantic:
  tool:
    - finance_.*
  prompt:
    - finance_.*
  resource:
    - .*

finance_registry:
  registry: "{username}"
  tool:
    - finance_.*
  prompt:
    - finance_.*
  resource:
    - .*
"""

with open(os.path.join(CONFIG_DIR, "profiles.yml"), "w", encoding="utf-8") as f:
    f.write(profiles_yaml)

print(profiles_yaml)

<hr style="height:2px;border:none">
<p style="font-size:20px; font-family:Arial">
  <b>4. Explore the Semantic Layer with Direct MCP Calls</b>
</p>

<p style="font-size:16px; font-family:Arial">
Before any LLM is involved, we exercise each object type by hand. This makes the server's behavior visible and
verifiable: what tool signature does a cube generate, what SQL discipline do the two filter phases impose, when
does the optional join materialize, and what happens on an invalid request.
</p>

<p style="font-size:18px; font-family:Arial"><b>4.1 Start the MCP with the <code>finance_semantic</code> profile</b></p>

In [ ]:
TD_HOST = env_vars["host"]
TD_USER = env_vars["username"]
TD_PASSWORD = env_vars["my_variable"]  
TD_DB = TD_USER

DATABASE_URI = f"teradata://{TD_USER}:{TD_PASSWORD}@{TD_HOST}:1025/{TD_DB}"

mcp_env = {
    "DATABASE_URI": DATABASE_URI,
    "MCP_TRANSPORT": "stdio",
}

client_sem = MultiServerMCPClient(
    {
        "teradata": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [
                "-m", "teradata_mcp_server",
                "--profile", "finance_semantic",
                "--config_dir", CONFIG_DIR,
            ],
            "env": mcp_env,
            "cwd": CONFIG_DIR,
        }
    }
)

mcp_tools = await client_sem.get_tools()
print([t.name for t in mcp_tools])

<p style="font-size:16px; font-family:Arial">
Three tools: the two YAML tools plus <code>finance_business_portfolio</code>. The cube <b>is</b> a tool: the
server generated it from the cube definition. Inspect its signature and description, which is what any MCP client
(and any LLM) sees:
</p>

In [ ]:
cube = next(t for t in mcp_tools if t.name == "finance_business_portfolio")
print("DESCRIPTION:\n", cube.description, "\n")
print("INPUT SCHEMA:")
print(json.dumps(cube.args, indent=2))

<p style="font-size:16px; font-family:Arial">
The generated signature follows the pattern from the Customizing guide: <code>dimensions</code> and
<code>measures</code> (comma-separated published names), <code>filter</code>, <code>res_filter</code>,
<code>order_by</code>, <code>top</code>, plus our custom <code>status_filter</code> parameter.
</p>

<p style="font-size:18px; font-family:Arial"><b>4.2 The server-side prompt object</b></p>

<p style="font-size:16px; font-family:Arial">
The <code>finance_portfolio_analyst</code> prompt is served by the MCP server, not hardcoded in a client. Any
client connecting to this server retrieves the same governed instructions. We fetch it through the MCP session and
will reuse it, verbatim, as the agent's system prompt in Section 5.
</p>
<div class="alert alert-block alert-info">
<p style="font-size:16px;font-family:Arial"><i><b>Note:</b> some server builds have a prompt rendering issue where <code>list_prompts</code> succeeds but <code>get_prompt</code> fails with "Prompt must return str, list[Message], or PromptResult, got Message". The cell below tries the MCP retrieval first and, if that fails, loads the identical governed definition from the server&rsquo;s own config file. Either way, the instructions come from server-side configuration, not from client code.</i></p>
</div>

In [ ]:
server_prompt = None

async with client_sem.session("teradata") as session:
    prompt_list = await session.list_prompts()
    print("Prompts served by the MCP server:")
    for p in prompt_list.prompts:
        print(f"  - {p.name}: {p.description}")

    try:
        fetched = await session.get_prompt("finance_portfolio_analyst")
        server_prompt = fetched.messages[0].content.text
        print("\nRetrieved via MCP get_prompt.")
    except Exception as e:
        # Known issue on some server builds: the dynamic prompt render function
        # returns a single Message where FastMCP requires str, list[Message]
        # or PromptResult, so get_prompt fails even though list_prompts works.
        print(f"\nMCP get_prompt not usable on this server build: {str(e)[:150]}")

# Fallback: read the same governed definition from the server's config file.
# The prompt still lives in server-side configuration; only the retrieval
# path changes.
if not server_prompt:
    import yaml
    with open(os.path.join(CONFIG_DIR, "finance_objects.yml"), "r", encoding="utf-8") as f:
        objects = yaml.safe_load(f)
    server_prompt = objects["finance_portfolio_analyst"]["prompt"]
    print("Loaded the prompt from the server's governed config (finance_objects.yml).")

print("\n--- finance_portfolio_analyst ---\n")
print(server_prompt)

<p style="font-size:18px; font-family:Arial"><b>4.3 The glossary, auto-enriched</b></p>

<p style="font-size:16px; font-family:Arial">
Glossary terms are exposed as MCP <b>resources</b>. The server automatically enriches each term with references to
the cubes and tools that relate to it, so the vocabulary stays connected to the objects that implement it.
</p>

In [ ]:
async with client_sem.session("teradata") as session:
    resources = await session.list_resources()
    print("Resources served by the MCP server:")
    for r in resources.resources:
        print(f"  - {r.name}  ({r.uri})")

    # Read the glossary term resources
    for r in resources.resources:
        try:
            content = await session.read_resource(r.uri)
            text = content.contents[0].text if content.contents else ""
            print(f"\n--- {r.name} ---")
            print(text[:600])
        except Exception as e:
            print(f"\n--- {r.name}: could not read ({e}) ---")

<p style="font-size:18px; font-family:Arial"><b>4.4 Calling the cube: the server compiles the SQL</b></p>

<p style="font-size:16px; font-family:Arial">
Each call below states an analytical intent in business vocabulary. The server compiles it into SQL against the
governed model. Start with the portfolio broken down by region:
</p>

In [ ]:
def show(res, label):
    print(f"--- {label} ---")
    print(res[0]["text"][:1200], "\n")

res = await cube.ainvoke({
    "dimensions": "region",
    "measures": "business_count, risk_flagged_count, base_eligible_count",
})
show(res, "Portfolio by region")

<p style="font-size:16px; font-family:Arial">
<b>Pre-aggregation filter.</b> <code>filter</code> reduces the base row set before <code>GROUP BY</code>, and may
reference any cube dimension even when it is not selected. Here we profile tenure by business type, restricted to
non-risk businesses:
</p>

In [ ]:
res = await cube.ainvoke({
    "dimensions": "business_type",
    "measures": "business_count, avg_tenure_months",
    "filter": "risk_flag = 'N'",
})
show(res, "Tenure by business type, risk-flagged excluded (pre-aggregation filter)")

<p style="font-size:16px; font-family:Arial">
<b>Post-aggregation filter.</b> <code>res_filter</code> runs after aggregation and references computed measures.
Only region and type segments with more than one business survive:
</p>

In [ ]:
res = await cube.ainvoke({
    "dimensions": "region, business_type",
    "measures": "business_count",
    "res_filter": "business_count > 1",
    "order_by": "business_count",
})
show(res, "Segments with more than one business (post-aggregation filter)")

<p style="font-size:16px; font-family:Arial">
<b>The optional join materializes only on demand.</b> Selecting <code>region_size</code> forces the compiler to
materialize the <code>region_totals</code> subquery join. None of the earlier calls paid for that join:
</p>

In [ ]:
res = await cube.ainvoke({
    "dimensions": "region, region_size",
    "measures": "business_count",
})
show(res, "Region size dimension (optional join materialized)")

<p style="font-size:16px; font-family:Arial">
<b>The nullable optional parameter.</b> Providing <code>status_filter</code> narrows the base row set; passing a
placeholder like <code>NULL</code> is treated as no filter:
</p>

In [ ]:
res = await cube.ainvoke({
    "dimensions": "region",
    "measures": "business_count",
    "status_filter": "Active",
})
show(res, "Active businesses only (custom optional parameter)")

<p style="font-size:16px; font-family:Arial">
<b>Invalid requests fail before SQL is generated.</b> This is a defining trust property of the cube: an
LLM that hallucinates a measure name gets an immediate, correcting error listing the allowed names, and the
database never sees the request. Compare this with an LLM writing raw SQL, where a hallucinated column either
errors cryptically or, worse, silently resolves to the wrong thing:
</p>

In [ ]:
try:
    res = await cube.ainvoke({
        "dimensions": "region",
        "measures": "total_revenue",   # not a published measure of this cube
    })
    show(res, "Unexpected success")
except Exception as e:
    print("Cube rejected the request before any SQL was sent to the database:\n")
    print(str(e)[:800])

<p style="font-size:16px; font-family:Arial">
That completes the tour of the YAML layer: one governed definition, a generated tool signature, deterministic SQL
compilation, and validation at the semantic boundary. Now hand it to a model.
</p>

<hr style="height:2px;border:none">
<p style="font-size:20px; font-family:Arial">
  <b>5. The Agent on Top of the Semantic Layer</b>
</p>

<p style="font-size:16px; font-family:Arial">
Notice how thin the agent becomes. Its system prompt is the one <b>retrieved from the server</b> in Section 4.2,
its tools are the semantic layer, and it contains zero domain knowledge of its own. Every piece of business logic
(metric definitions, eligibility clause, vocabulary) lives in the YAML file, where the data team can review,
version and change it.
</p>

<p style="font-size:18px; font-family:Arial"><b>5.1 Initialize the LLM</b></p>

In [ ]:
from langchain.chat_models import init_chat_model
llm_key = env_vars.get("litellm_key")
llm_url = env_vars.get("litellm_base_url")
model_name = env_vars.get("reasoning_openai_primary")

llm = init_chat_model(
    model=model_name,
    model_provider="openai",
    base_url=llm_url,
    api_key=llm_key,
)

<p style="font-size:18px; font-family:Arial"><b>5.2 Create the agent with the server-served prompt</b></p>

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=mcp_tools,          # the semantic layer: two tools + the cube
    system_prompt=server_prompt,   # retrieved from the MCP server in Section 4.2
)

def final_answer(result):
    for m in reversed(result.get("messages", [])):
        if getattr(m, "type", "") == "ai" and isinstance(m.content, str) and m.content.strip():
            return m.content.strip()
    return "(No final answer returned.)"

def tool_trace(result):
    return [
        {"tool": getattr(m, "name", "tool"), "output": str(m.content)[:200]}
        for m in result.get("messages", []) if getattr(m, "type", "") == "tool"
    ]

<p style="font-size:18px; font-family:Arial"><b>5.3 Ask portfolio questions in plain language</b></p>

<p style="font-size:16px; font-family:Arial">
The first question requires an aggregate comparison, exactly what the cube exists for. The model's job reduces to
translating the question into published dimension and measure names; the server does the rest deterministically.
</p>

In [ ]:
question = ("Which region has the highest share of risk-flagged businesses, "
            "and how many businesses in that region pass the base loyalty eligibility clause?")

result = await agent.ainvoke({"messages": [HumanMessage(content=question)]})

for t in tool_trace(result):
    print(f"--- TOOL: {t['tool']} ---")
    print(t['output'], "\n")

print("=" * 60)
print(final_answer(result))

In [ ]:
question = "Compare average tenure between Enterprise and SMB businesses, excluding risk-flagged accounts."

result = await agent.ainvoke({"messages": [HumanMessage(content=question)]})

for t in tool_trace(result):
    print(f"--- TOOL: {t['tool']} ---")
    print(t['output'], "\n")

print("=" * 60)
print(final_answer(result))

<p style="font-size:16px; font-family:Arial">
In both traces the agent called the cube with published names, and the answers cite them, as the governed prompt
requires. If the model ever requests a metric that does not exist, Section 4.4 showed what happens: a correcting
error at the semantic boundary, not a guess against your schema.
</p>

<hr style="height:2px;border:none">
<p style="font-size:20px; font-family:Arial">
  <b>6. The Database Tool Registry: Tools that Live with the Data</b>
</p>

<p style="font-size:16px; font-family:Arial">
YAML files are one way to define tools. The <b>Database Tool Registry</b> is the alternative for organizations
that want tool definitions to live <i>inside the database</i>, next to the objects they wrap, governed by database
permissions. The server reads two views from a designated registry schema:
</p>

<ul style="font-size:16px; font-family:Arial; margin-left:20px; line-height:1.8;">
  <li><code>mcp_list_tools</code>: one row per tool (name, target object, type code <code>F</code>/<code>M</code>/<code>T</code>/<code>V</code>, docstring, tags)</li>
  <li><code>mcp_list_toolParams</code>: one row per parameter (name, Teradata type code, position, required flag, comment)</li>
</ul>

<p style="font-size:16px; font-family:Arial">
Registry tools load on the first database connection and are always loaded in full: profile <code>tool</code>
patterns apply only to code-based tools. Access control instead happens <b>in the database</b>: the
<code>mcp_list_tools</code> view filters with <code>WHERE username = USER</code> against a
<code>mcp_tool_user_access</code> mapping table, so a shared registry serves different tool sets to different
connecting users. That is the per-user filtering pattern from the Customizing guide.
</p>

<p style="font-size:16px; font-family:Arial">
We follow the repository's
<a href="https://github.com/Teradata/teradata-mcp-server/blob/main/examples/server-customisation/registry_setup.sql" target="_blank">registry setup example</a>,
adapted in one way: the example creates a dedicated <code>mcp</code> database, which demo users typically cannot
do, so we build the registry in <b>your own database</b> and point the profile's <code>registry</code> key at it
(already done in Section 3.3).
</p>

<p style="font-size:18px; font-family:Arial"><b>6.1 Create the registry infrastructure</b></p>

<p style="font-size:16px; font-family:Arial">
Base tables for registrations, user access and tags, then the layered views ending in the two views the server
reads. Descriptions and parameter documentation come from database <code>COMMENT</code> strings via
<code>DBC.TablesV</code> and <code>DBC.ColumnsV</code>, so documenting the database object documents the tool.
</p>

In [ ]:
registry_ddl = [
    # ---- base tables ----
    """CREATE TABLE mcp_tool(
        ToolName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NOT NULL,
        DataBaseName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NOT NULL,
        TableName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NULL,
        description VARCHAR(10000) CHARACTER SET UNICODE,
        registered_ts TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    ) UNIQUE PRIMARY INDEX(ToolName);""",
    """CREATE TABLE mcp_tool_user_access(
        ToolName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NOT NULL,
        UserName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NOT NULL
    ) PRIMARY INDEX (ToolName);""",
    """CREATE TABLE mcp_tool_tag(
        ToolName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NOT NULL,
        TagName VARCHAR(128) CHARACTER SET UNICODE NOT CASESPECIFIC NOT NULL
    ) PRIMARY INDEX (ToolName);""",
    # ---- layered views ----
    """REPLACE VIEW mcp_toolV AS
    SELECT
        r.ToolName,
        r.DataBaseName,
        r.TableName,
        COALESCE(r.description, t.CommentString) AS description,
        t.TableKind AS toolType,
        r.registered_ts
    FROM mcp_tool r
    JOIN DBC.TablesV t
        ON r.DataBaseName = t.DataBaseName
        AND r.TableName = t.TableName
    WHERE t.TableKind IN ('M', 'V', 'T', 'F');""",
    """REPLACE VIEW mcp_toolParamsV AS
    SELECT
        r.ToolName,
        c.ColumnName AS ParamName,
        TRIM(c.ColumnType) AS ParamType,
        c.ColumnLength AS ParamLength,
        c.ColumnId - 1024 AS ParamPosition,
        c.Nullable AS ParamRequired,
        c.CommentString AS ParamComment
    FROM mcp_tool r
    JOIN DBC.TablesV t
        ON r.DataBaseName = t.DataBaseName
        AND r.TableName = t.TableName
    JOIN DBC.ColumnsV c
        ON c.DatabaseName = t.DatabaseName
        AND c.TableName = t.TableName
    WHERE t.TableKind <> 'F' OR c.SPParameterType = 'I';""",
    """REPLACE VIEW mcp_toolDocV AS
    SELECT
        t.toolName,
        COALESCE(t.description, 'No description') || CHR(10) ||
        CHR(10) ||
        'Arguments:' || CHR(10) || '   ' ||
        COALESCE(
            (SELECT TRIM(BOTH FROM
                XMLAGG(
                    '  ' || TRIM(p.ParamName) || ' - ' ||
                    COALESCE(p.ParamComment, 'no description') || CHR(10)
                    ORDER BY p.ParamPosition
                ) (VARCHAR(10000))
            )
             FROM mcp_toolParamsV p
             WHERE p.ToolName = t.ToolName),
            '  (no parameters)'
        ) AS docstring
    FROM mcp_toolV t;""",
    # ---- the two views the MCP server reads ----
    """REPLACE VIEW mcp_list_tools AS
    SELECT
        t.ToolName,
        r.DataBaseName,
        r.TableName,
        TRIM(r.toolType) AS toolType,
        r.registered_ts,
        d.docstring,
        XMLAGG(tt.TagName || ' ') AS Tags
    FROM mcp_toolV r
    JOIN mcp_tool_user_access t
        ON t.ToolName = r.ToolName
    LEFT JOIN mcp_tool_tag tt
        ON t.ToolName = tt.ToolName
    LEFT JOIN mcp_toolDocV d
        ON t.ToolName = d.ToolName
    WHERE t.UserName = USER
    GROUP BY 1, 2, 3, 4, 5, 6;""",
    """REPLACE VIEW mcp_list_toolParams AS
    SELECT
        r.*
    FROM mcp_toolParamsV r
    JOIN mcp_tool_user_access t
        ON t.ToolName = r.ToolName
    WHERE t.UserName = USER;""",
]

for stmt in registry_ddl:
    try:
        execute_sql(stmt)
    except Exception as e:
        # CREATE TABLE fails if it already exists from a prior run; that's fine
        print(f"Note: {str(e)[:120]}")

print("Registry tables and views are in place.")

<p style="font-size:18px; font-family:Arial"><b>6.2 Create the database object the tool wraps</b></p>

<p style="font-size:16px; font-family:Arial">
A Teradata <b>macro</b>, <code>finance_business_profile</code>, wrapping the eligibility query. The
<code>COMMENT</code> statements are not decoration: the registry views turn the macro comment into the tool's
docstring and the parameter comment into the parameter's description, which is what the LLM will read.
</p>

In [ ]:
execute_sql("""
REPLACE MACRO finance_business_profile(
    business_name VARCHAR(100)
) AS (
    SELECT
        business_id,
        business_name,
        business_type,
        region,
        status,
        risk_flag,
        ((EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM start_date)) * 12
          + (EXTRACT(MONTH FROM CURRENT_DATE) - EXTRACT(MONTH FROM start_date))) AS tenure_months
    FROM "DEMO_Financial"."Business"
    WHERE business_name = :business_name;
);
""")

execute_sql("""
COMMENT ON MACRO finance_business_profile AS
'Return the full profile of one business (type, region, status, risk flag and tenure in months) for loyalty discount evaluation.';
""")

execute_sql("""
COMMENT ON COLUMN finance_business_profile.business_name AS 'Exact business_name as stored in DEMO_Financial.Business.';
""")

print("Macro finance_business_profile created and documented.")

<p style="font-size:18px; font-family:Arial"><b>6.3 Register the tool and grant user access</b></p>

<p style="font-size:16px; font-family:Arial">
Registration is an <code>INSERT</code>. Access is a row in <code>mcp_tool_user_access</code>: the per-user
filtering lever. A user with no row sees no tool, from the same shared registry.
</p>

In [ ]:
execute_sql("DELETE FROM mcp_tool WHERE ToolName = 'finance_business_profile';")
execute_sql("DELETE FROM mcp_tool_user_access WHERE ToolName = 'finance_business_profile';")
execute_sql("DELETE FROM mcp_tool_tag WHERE ToolName = 'finance_business_profile';")

execute_sql(f"""
INSERT INTO mcp_tool(ToolName, DataBaseName, TableName)
VALUES ('finance_business_profile', '{username}', 'finance_business_profile');
""")

# Grant access to the current database user (the per-user filtering lever)
execute_sql("""
INSERT INTO mcp_tool_user_access
SELECT ToolName, USER FROM mcp_toolV WHERE ToolName = 'finance_business_profile';
""")

execute_sql("INSERT INTO mcp_tool_tag VALUES ('finance_business_profile', 'finance');")

print("Tool registered and access granted to the current user.")

<p style="font-size:18px; font-family:Arial"><b>6.4 Verify what the server will see</b></p>

<p style="font-size:16px; font-family:Arial">
These are exactly the two queries the registry loader runs on first connection. Note the docstring assembled from
the macro and parameter comments:
</p>

In [ ]:
print("mcp_list_tools:")
cur = execute_sql("SELECT ToolName, DataBaseName, TableName, toolType, docstring FROM mcp_list_tools;")
for row in cur.fetchall():
    print(f"  ToolName={row[0]}  db={row[1]}  object={row[2]}  type={row[3]}")
    print(f"  docstring:\n{row[4]}")
cur.close()

print("\nmcp_list_toolParams:")
cur = execute_sql("SELECT ToolName, ParamName, ParamType, ParamPosition, ParamRequired, ParamComment FROM mcp_list_toolParams;")
for row in cur.fetchall():
    print(f"  {row}")
cur.close()

<p style="font-size:18px; font-family:Arial"><b>6.5 Restart the MCP with the <code>finance_registry</code> profile</b></p>

<p style="font-size:16px; font-family:Arial">
Same YAML semantic layer, plus the registry. The new tool appears alongside the YAML tools, loaded from the
database, and the server executes it as <code>EXEC {your_db}.finance_business_profile(:business_name)</code> with
safe named-parameter binding.
</p>

In [ ]:
client_reg = MultiServerMCPClient(
    {
        "teradata": {
            "transport": "stdio",
            "command": sys.executable,
            "args": [
                "-m", "teradata_mcp_server",
                "--profile", "finance_registry",     # <-- profile with the registry key
                "--config_dir", CONFIG_DIR,
            ],
            "env": mcp_env,
            "cwd": CONFIG_DIR,
        }
    }
)

registry_tools = await client_reg.get_tools()
print([t.name for t in registry_tools])

In [ ]:
profile_tool = next(t for t in registry_tools if t.name == "finance_business_profile")
print("Tool description as the LLM sees it (from database COMMENTs):\n")
print(profile_tool.description, "\n")

res = await profile_tool.ainvoke({"business_name": "Acme Co"})
print(res[0]["text"][:600])

<p style="font-size:16px; font-family:Arial">
The same agent pattern from Section 5 works unchanged on this expanded tool set; only the tool's <i>definition
home</i> moved, from a YAML file to the database. Choose YAML when data teams own the semantic layer as
version-controlled configuration; choose the registry when tools should live and be governed alongside the
database objects they wrap. The two coexist in one profile, as this section just demonstrated.
</p>

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>7. Cleanup</b></p>
<p style = 'font-size:18px;font-family:Arial'><b>7.1 Drop the registry objects and the macro</b></p>

In [ ]:
cleanup_stmts = [
    "DROP VIEW mcp_list_tools;",
    "DROP VIEW mcp_list_toolParams;",
    "DROP VIEW mcp_toolDocV;",
    "DROP VIEW mcp_toolParamsV;",
    "DROP VIEW mcp_toolV;",
    "DROP TABLE mcp_tool_tag;",
    "DROP TABLE mcp_tool_user_access;",
    "DROP TABLE mcp_tool;",
    "DROP MACRO finance_business_profile;",
]

for stmt in cleanup_stmts:
    try:
        execute_sql(stmt)
        print(f"OK: {stmt}")
    except Exception as e:
        print(f"Skipping ({stmt}): {str(e)[:100]}")

<p style = 'font-size:18px;font-family:Arial'><b>7.2 Close the connection</b></p>

In [ ]:
remove_context()

<footer style="padding-bottom:35px; border-bottom:3px solid">
      <div style="float:right;">
        <div style="float:left; margin-top:14px">
            Copyright © Teradata - 2026. All Rights Reserved
        </div>
    </div>
</footer>